In [1]:
# ==============================================================================
# PIPELINE 6A: Research Question 1 - Unsupervised Market Clustering
# Objective: Identify hidden structural archetypes in European public procurement.
# ==============================================================================
import pandas as pd
from src.training.preparation import prepare_clustering_data
from src.training.optimization import find_optimal_clusters
from src.training.evaluation import analyze_cluster_profiles

# Configure pandas for better dataframe visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [2]:
# ------------------------------------------------------------------------------
# STEP 1: Load Data & Select Relevant Structural Features
# ------------------------------------------------------------------------------
print("Loading prepared dataset...")
dataset_prepared = pd.read_parquet("data/prepared/ted_prepared.parquet")

# Select features that define the 'shape' and 'complexity' of a project,
# explicitly excluding outcomes (Prices, Tenders) to find pure planning archetypes.
clustering_features = [
    'DURATION', 
    'PREPARATION_DAYS', 
    'IS_CONTRACT_W', 
    'IS_CONTRACT_S', 
    'IS_CONTRACT_U', 
    'B_EU_FUNDS', 
    'IS_FRAMEWORK', 
    'NUTS_REGION_COUNT'
]

Loading prepared dataset...


In [3]:
# ------------------------------------------------------------------------------
# STEP 2: Standardize Data for Distance-Based Algorithms
# ------------------------------------------------------------------------------
# K-Means calculates geometric distances. We MUST scale the data so that 
# 'DURATION' (e.g., 365 days) doesn't completely overpower 'IS_FRAMEWORK' (0 or 1).
features_scaled, scaling_pipeline = prepare_clustering_data(
    df=dataset_prepared, 
    features=clustering_features
)

Preparing features for Unsupervised Clustering (Imputation & Scaling)...
 -> Scaled 8 features for 2,425,527 observations.


In [4]:
# ------------------------------------------------------------------------------
# STEP 3: Hyperparameter Search (Finding optimal K via Silhouette Score)
# ------------------------------------------------------------------------------
# This tests k=2 through k=8 to find the mathematically most distinct groups.
optimal_kmeans_model = find_optimal_clusters(X_scaled=features_scaled, max_k=6)

# Assign the discovered cluster labels back to our original dataset
dataset_prepared['MARKET_ARCHETYPE_ID'] = optimal_kmeans_model.labels_

\nSearching for optimal market clusters (K=2 to 6)...
 -> K=2: Silhouette Score = 0.5339
 -> K=3: Silhouette Score = 0.2490
 -> K=4: Silhouette Score = 0.2933
 -> K=5: Silhouette Score = 0.4131
 -> K=6: Silhouette Score = 0.4101
\nOptimal Market Archetypes found: K=2 (Score: 0.5339)


In [5]:
# ------------------------------------------------------------------------------
# STEP 4: Profile Analysis (Interpreting the Archetypes)
# ------------------------------------------------------------------------------
# We analyze the median values of the original unscaled data to understand 
# what real-world characteristics define each cluster.
analyze_cluster_profiles(
    df_original=dataset_prepared[clustering_features], 
    cluster_labels=optimal_kmeans_model.labels_
)

\nMarket Archetype Profiles (Median Feature Values):


Market_Archetype,0,1
DURATION,3.2189,3.2189
PREPARATION_DAYS,3.5835,3.6109
IS_CONTRACT_W,0.0000,0.0000
IS_CONTRACT_S,0.0000,0.0000
IS_CONTRACT_U,0.0000,1.0000
B_EU_FUNDS,0.0000,0.0000
IS_FRAMEWORK,0.0000,0.0000
NUTS_REGION_COUNT,1.0986,0.6931
